# BirdCLEF+ 2025

## Import

In [2]:
import numpy as np
import os
import librosa
from scipy.io import wavfile
from torch.utils.data import DataLoader, Dataset, random_split
import torch
import time


## Custom dataset

In [5]:
class AudioDataset(Dataset):
    def __init__(self, root_path="/data/in/birdclef-2025/train_audio", active='wavs', normalize=False):
        self.max_length = 0
        self.class_map = {"esben" : 0, "peter": 1, "both": 2}
        self.data = []
        self.labels = []
        self.min_val = 10e10
        self.max_val = 0
        self.normalize = normalize
        self.wavs, self.mfccs, self.chromas, self.contrasts, self.centroids, self.bandwidths = [], [], [], [], [], []
        self.wavs_norm, self.mfccs_norm, self.chromas_norm, self.contrasts_norm, self.centroids_norm, self.bandwidths_norm = [], [], [], [], [], []
        
        print("Start reading files and genearting features")
        
        for specie in os.listdir(root_path):
            label = specie
            for file in os.listdir(os.path.join(root_path, specie)):
                start = time.time()
                wav, sample_rate = librosa.load(os.path.join(root_path, specie, file), sr=None)
                print(f"{time.time()-start:05.2f} s")
                wav = wav.astype(np.float32)
                start = time.time()
                print(len(wav))
                print(f"{time.time()-start:05.2f} s")
                return
                
            
                # if wav.shape[0] > self.max_length:
                #     self.max_length = wav.shape[0]
                #     print("Found wav with more length than specified max one, new max is:", wav.shape[0])
                
                # self.feature_extraction(wav, self.sample_rate)
                # wav = np.pad(wav, (0, self.max_length-wav.shape[0]))
                # label_str = file_path.split('/')[-3][2:]
                # label = (np.int64(self.class_map[label_str]))
                
                # self.max_val = np.max(wav) if np.max(wav) > self.max_val else self.max_val
                # self.min_val = np.min(wav) if np.min(wav) < self.min_val else self.min_val
                
                # self.wavs.append(wav)
                # self.labels.append(label)
               
        self.wavs = np.array(self.wavs)
        self.mu  = self.wavs.mean()
        self.std = np.std(self.wavs)
        # self.wavs = torch.Tensor(self.wavs)

        self.active = active
        self.values_dict = {'wavs': 0, 'mfcc': 1, 'chroma': 2, 'contrast': 3, 'centroid': 4, 'bandwidth': 5}
        self.values_list = [self.wavs, self.mfccs, self.chromas, self.contrasts, self.centroids, self.bandwidths]
        self.values_norm_list = []
        print("Generating normalized arrays")
        for lst in self.values_list:
            self.values_norm_list.append((lst + np.abs(np.min(lst))) / (np.abs(np.min(lst)) + np.max(lst)))
            
        print("="*40)
        print("Loaded DATABASE from {}\n{} total file\nLongest file is {} long\nMean: {}\nStandard deviation: {}\n".
              format(root_path, len(self.wavs), self.max_length, self.mu, self.std))
        print("="*40)

    def feature_extraction(self, wav, sample_rate):
        self.mfccs.append(np.transpose(np.mean(librosa.feature.mfcc(y=wav, sr=sample_rate, n_mfcc=128).T, axis=0)))
        # self.chromas.append(np.transpose(np.mean(librosa.feature.chroma_cqt(y=wav, sr=sample_rate).T, axis=0)))
        self.chromas.append(np.transpose(np.mean(librosa.feature.chroma_stft(y=wav, sr=sample_rate).T, axis=0)))
        self.contrasts.append(np.transpose(np.mean(librosa.feature.spectral_contrast(y=wav, sr=sample_rate).T, axis=0)))
        self.centroids.append(np.transpose(np.mean(librosa.feature.spectral_centroid(y=wav, sr=sample_rate).T, axis=0)))
        self.bandwidths.append(np.transpose(np.mean(librosa.feature.spectral_bandwidth(y=wav, sr=sample_rate).T, axis=0)))

    def __len__(self):
        return len(self.wavs)
    
    def __getitem__(self, idx):
        y = self.labels[idx]
        x = self.values_list[self.values_dict[self.active]][idx]
        if self.normalize:
            x = self.values_norm_list[self.values_dict[self.active]][idx]
        x = torch.Tensor(x)
        return x, y
    
dt = AudioDataset("../data/in/birdclef-2025/train_audio")

Start reading files and genearting features
00.04 s
3450155
00.00 s
